In [ ]:
!curl https://raw.githubusercontent.com/pytorch/xla/master/contrib/scripts/env-setup.py -o pytorch-xla-env-setup.py
!python pytorch-xla-env-setup.py --version 1.7 --apt-packages libomp5 libopenblas-dev

In [ ]:
! pip install pytorch-lightning==1.1.8

In [ ]:
! pip install lightning-bolts

In [ ]:
! pip install pytorch-metric-learning
! pip install timm

In [ ]:
!pip install wandb --upgrade

In [ ]:
pip install --upgrade albumentations

# Load libraries

In [ ]:
import platform
import numpy as np 
import pandas as pd 
import os
from tqdm.notebook import tqdm
import cv2
# import pydicom
import random
# import gdcm
import glob
import gc
from math import ceil
import matplotlib.pyplot as plt
# from pydicom.pixel_data_handlers.util import apply_voi_lut

import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

import albumentations

# from sklearn.model_selection import StratifiedKFold, GroupKFold

import math
import psutil

DATA_PATH = '/kaggle/input/siim-covid19-detection/'

In [ ]:
from PIL import Image

import torch
torch.manual_seed(0)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

import timm

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from pytorch_metric_learning import losses
from torch.optim import lr_scheduler

import warnings
warnings.simplefilter('ignore')

In [ ]:
import pytorch_lightning as pl
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.metrics.functional import accuracy
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

from torch.optim import Adam
from torch.nn.functional import cross_entropy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pl.seed_everything(101)

# Config

In [ ]:
Config = dict (
    train_pcent = 0.85,
    folds = 5,
    model_name = 'tf_efficientnet_b6_ns',
    image_size = (512, 512),
    num_classes = 4,
    batch_size = 32,
    num_workers = 8,
    num_epochs = 25,
    scaler = GradScaler(),
    scheduler = 'CosineAnnealingLR', # CosineAnnealingLR, ReduceLROnPlateau, CosineAnnealingWarmRestarts, CyclicLR
    weight_decay = 1e-6,
    T_0 = 5, # CosineAnnealingWarmRestarts
    T_mult = 2, # CosineAnnealingWarmRestarts
    T_max = 10,
    eta_min = 1e-5,
    lr = 0.01,
    factor=0.3, # ReduceLROnPlateau
    patience=3, # ReduceLROnPlateau
    eps=1e-7, # ReduceLROnPlateau
    accumulate_grad = 4 #{5: 4, 15: 8, 35: 20}
)

# Custom functions

In [ ]:
# Function to store pickle object
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
# Function to load pickle object
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

# Prepare dataset

In [ ]:
class SIIMData(Dataset):
    def __init__(self, df, is_train=True, augments=None, img_size=Config['image_size']):
        super().__init__()
        self.df = df.sample(frac=1).reset_index(drop=True)
        self.is_train = is_train
        self.augments = augments
        self.img_size = img_size
        
    def __getitem__(self, idx):
        image_id = self.df['StudyInstanceUID'].values[idx]
        
        image_path = self.df['path'].values[idx]
        image = cv2.imread(image_path)

        # image = image.astype(np.int)
        
        # Augments must be albumentations
        if self.augments:
            image = self.augments(image=image)['image']
        
        image = torch.tensor(image)
        
        if self.is_train:
            label = self.df[self.df['StudyInstanceUID'] == image_id].values.tolist()[0][5:9]#.index(1)
            return image, torch.tensor(label)
        
        return image
    
    def __len__(self):
        return len(self.df)

In [ ]:
transforms_train = albumentations.Compose([
    albumentations.OneOf([
                            albumentations.HueSaturationValue(hue_shift_limit=0.2, sat_shift_limit= 0.2, 
                                                val_shift_limit=0.2, p=0.9),
                            albumentations.RandomBrightnessContrast(brightness_limit=0.2, 
                                                    contrast_limit=0.2, p=0.9),
                        ],p=0.5),
#     albumentations.RandomCrop(width=256, height=256, p=1),
    albumentations.VerticalFlip(p=0.3),
    albumentations.HorizontalFlip(p=0.5),
    albumentations.Rotate(p=0.5),
    albumentations.ImageCompression(quality_lower=85, quality_upper=95, p=0.5),
#     albumentations.GaussianBlur(p=0.3),
#     albumentations.Affine(p=0.3),
    albumentations.OneOf([
                            albumentations.Blur(blur_limit=3, p=1.0), 
                            albumentations.MedianBlur(blur_limit=3, p=1.0),
                            albumentations.MotionBlur(p=1)], p=0.5),
#     albumentations.RandomBrightnessContrast(p=0.5),
    albumentations.CoarseDropout(p=0.3, max_height=round(Config['image_size'][0]*0.1), 
                                 max_width=round(Config['image_size'][0]*0.1)),
    albumentations.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
])

transforms_valid = albumentations.Compose([
    albumentations.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
])

transforms_test = albumentations.Compose([
    albumentations.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
])

# Dataloader testing

In [ ]:
training_set = SIIMData(
                    df=train_df[train_df['fold']!=0],
                    augments=transforms_train
                )

validation_set = SIIMData(
                    df=train_df[train_df['fold']==0],
                    augments=transforms_valid
                )

In [ ]:
train_dataloader = DataLoader(training_set, batch_size=Config['batch_size'], shuffle=False, num_workers=8, pin_memory=True)
val_dataloader = DataLoader(validation_set, batch_size=Config['batch_size'], shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
for batch in train_dataloader:
    x, y = batch 
    break

In [ ]:
x.shape

In [ ]:
y.shape

# Pytorch lightning

## Lit DataModule

In [ ]:
class SIIMCovid19DataModule(pl.LightningDataModule):
    def __init__(self, batch_size, fold):

        super().__init__()

        self.batch_size = batch_size
        self.fold = fold
        self.train_kfold = train_df

    def prepare_data(self):
        
        pass

        # print("Data Prep completed")
         
    def setup(self, stage=None):
        
        self.training_set = SIIMData(
                                df=self.train_kfold[self.train_kfold['fold']!=self.fold],
                                augments=transforms_train
                            )

        self.validation_set = SIIMData(
                                    df=self.train_kfold[self.train_kfold['fold']==self.fold],
                                    augments=transforms_valid
                                )
        
        # print("Set up completed")


    def train_dataloader(self):
        return DataLoader(self.training_set, batch_size=self.batch_size, shuffle=False, num_workers=8, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.validation_set, batch_size=self.batch_size, shuffle=False, num_workers=8, pin_memory=True)

## Lit Model

In [ ]:
class LitEffNetB6(pl.LightningModule):
    def __init__(self, model_name, num_classes, lr, weight_decay, fold):
        
        super().__init__()
        self.save_hyperparameters()
        
        # Model architecture
        self.backbone = timm.create_model(self.hparams.model_name, pretrained=True, in_chans=3, num_classes=self.hparams.num_classes)
        # self.backbone = timm.create_model("efficientnet_b3a", pretrained=True, in_chans=3)
        # self.finetune_layer = nn.Linear(512, self.hparams.num_classes)
        
        # compute the accuracy -- no need to roll your own!
        self.train_preds = []
        self.train_y = []
        
        self.mAP_best = 0
        
            
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config['T_max'], eta_min=Config['eta_min'])
        # scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=Config['factor'], patience=Config['patience'], verbose=True, eps=Config['eps'])
#         scheduler = lr_scheduler.CyclicLR(optimizer, cycle_momentum=False, base_lr=0.01, max_lr=0.1)
        
        # scheduler = lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=Config['T_0'], 
        #                                                      T_mult=Config['T_mult'], 
        #                                                      eta_min=Config['eta_min'], verbose=True)
        
        lr_scheduler_lit = {
            'scheduler': scheduler,
            'name': 'CosineAnnealingLR'
        }

        # lr_scheduler_lit = {
        #     'scheduler': scheduler,
        #     'monitor': 'val/mAP',
        #     'name': 'ReduceLROnPlateau'
        # }

        return [optimizer], [lr_scheduler_lit]

    
    def forward(self, x, y=None):
        
        x = x.to(self.device).float()
        x = x.permute(0, 3, 1, 2)

        preds = self.backbone(x)
        # preds = self.finetune_layer(features)

        return preds

    
    def training_step(self, batch, batch_idx):
        
        # print("Training step starts")
        
        x, y = batch
        
        x = x.to(self.device).float()
        y = y.to(self.device).float()
        x = x.permute(0, 3, 1, 2)

        preds = self.backbone(x)
        # preds = self.finetune_layer(features)
        
        BCEloss = criterion1(preds, y)
        # Create logs
        self.log('train/BCEloss', BCEloss)
        
        preds = nn.functional.log_softmax(preds)

        self.train_preds.extend(preds.cpu().detach().numpy())
        self.train_y.extend(y.cpu().detach().numpy())
        
        # print("Training step ends")
        
        return BCEloss#, preds, y
    
        
    
    def validation_step(self, batch, batch_idx):
        
        # print("Validation step starts")
        
        x, y = batch
        
        x = x.to(self.device).float()
        y = y.to(self.device).float()
        x = x.permute(0, 3, 1, 2)

        preds = self.backbone(x)
        # preds = self.finetune_layer(features)
        
        BCEloss = criterion1(preds, y)
        # Create logs
        self.log('val/BCEloss', BCEloss)
        
        preds = nn.functional.log_softmax(preds)
        
        # print("Validation step ends")

        return BCEloss, preds, y
    


    def training_epoch_end(self, training_step_outputs):
        """Called when the validation epoch ends."""
        
        # print("Validation epoch end starts")
        
        epoch = self.trainer.current_epoch    
        
        # Calculate mAP score
        mAPloss = average_precision_score(self.train_y, self.train_preds)
        # Create logs
        self.log('train/mAP', mAPloss)
        self.log('train/mAP(2/3)', (2*mAPloss)/3)


        
    
    def validation_epoch_end(self, val_step_outputs):
        """Called when the validation epoch ends."""
        
        # print("Validation epoch end starts")
        
        epoch = self.trainer.current_epoch
        
        preds = []
        y_true = []
        for out in val_step_outputs:
            _, pred, y = out
            preds.extend(pred.cpu().numpy())
            y_true.extend(y.cpu().numpy())
            
        
        # Calculate mAP score
        mAPloss = average_precision_score(y_true, preds)
        # Create logs
        self.log('val/mAP', mAPloss)
        self.log('val/mAP(2/3)', (2*mAPloss)/3)
        
        # # Histogram of logits
        self.logger.experiment.log(
        {"val/logits": wandb.Histogram(preds),
         "global_step": self.global_step})
        
        
        if mAPloss > self.mAP_best:
            print(f"Best mAP (Illi): {mAPloss}")
            self.mAP_best = mAPloss 
        
        
            # Save latest model checkpoint
            print(f"Saving latest model checkpoint at epoch {epoch}; mAP score: {mAPloss:.4f}; mAP(2/3) score: {(2*mAPloss)/3:.4f}")
            filename = f'/workspace/SIIM_COVID19/models/tf_efficientnet_b6_ns-siim-covid19-{epoch}-{mAPloss:.4f}-{self.hparams.fold}.ckpt'
            trainer.save_checkpoint(filename)

            # # Save model as Model Artifact
            artifact = wandb.Artifact(name=f'tf_efficientnet_b6_ns-{epoch}-{mAPloss:.4f}-{self.hparams.fold}', type='model')
            artifact.add_file(filename)
            run.log_artifact(artifact)

            ! rm {filename}
        
        # print("Validation epoch end ends")
        
            
    
    def on_save_checkpoint(self, checkpoint):
        pass
            
    
    def predict(self, batch, batch_idx):
        x = batch
        output = self(x, y=None)
        return output

# Training

In [ ]:
criterion1 = nn.BCEWithLogitsLoss()

# Training KFold

for fold in range(Config['folds']):

    print(f"Training fold {fold} starts")

    classifier = LitEffNetB6(model_name = Config['model_name'], 
                         num_classes = Config['num_classes'], 
                         lr = Config['lr'], 
                         weight_decay = Config['weight_decay'],
                         fold = fold)
    
    early_stop_callback = EarlyStopping(
                           monitor='val/mAP',
                           min_delta=0.00,
                           patience=5,
                           verbose=True,
                           mode='max'
                        )

    lr_monitor = LearningRateMonitor(logging_interval='step')
    
    project = 'kaggle-siim-covid'
    group = 'Study Classification'
    experiment = 'tf_efficientnet_b6_ns_512px_5fold'
    notes = f'tf_efficientnet_b6_ns; 512px images; fold {fold}; BCE, mAP, logit hist; CosineAnnealingLR'

    # Initialize W&B run
    run = wandb.init(project=project, 
                    config=Config,
                    group=group, 
                    job_type='train',
                    tags=['train', 'tf_efficientnet_b6_ns'],
                    notes=notes)
    
    # Define a WandbLogger
    wandb_logger = WandbLogger(project=project,
                            group=group,
                            notes=notes)
    
    # Data module
    dm = SIIMCovid19DataModule(Config['batch_size'], fold)

    # Define trainer
    trainer = pl.Trainer(progress_bar_refresh_rate=5, gpus=1, 
                        max_epochs=Config['num_epochs'], 
                        check_val_every_n_epoch = 1,
                        stochastic_weight_avg=True, 
#                         limit_train_batches = 5,
#                         limit_val_batches = 5,
                        accumulate_grad_batches=Config['accumulate_grad'],
                        callbacks=[early_stop_callback, lr_monitor],
                        amp_level = 'O1',
                        logger=wandb_logger
#                          fast_dev_run=True
                        )
    
    # Fit model
    trainer.fit(classifier, datamodule=dm)

    # End W&B run
    wandb.finish()